In [2]:
from coffea_workflow import Step, Workflow, Fileset, Analysis, Plotting, RunConfig, ExecutorConfig, run, CustomArtifact
from coffea_workflow import facilities
from coffea_workflow.facilities import CoffeaCasaFactory
from analysis import get_fileset, run_analysis, plot_results, custom_function_remove_last_file

step_fileset = Step(
    name="Fileset",
    step_type=Fileset,
    builder=get_fileset,
    builder_params={"to_print": "\nTEST:\nparameter testing...\nSUCCESS!\n"}
)

step_custom_filtering = Step(
    name="FilesetFiltering",
    step_type=CustomArtifact,
    builder=custom_function_remove_last_file,
)

step_analysis = Step(
    name="SingleMuonAnalysis",
    step_type=Analysis,
    builder=run_analysis,
)

step_plotting = Step(
    name="PlottingMuonAnalysis",
    step_type=Plotting,
    builder="analysis:plot_results"
)

workflow = Workflow()
workflow.add(step_fileset)
workflow.add(step_analysis, depends_on=[step_fileset])
workflow.add(step_plotting, depends_on=[step_analysis])

# # local, FuturesExecutor
# config = RunConfig(
#     strategy="by_dataset",
#     percentage=10,
#     chunk_fraction=0.4,
#     facility=facilities.local,
#     executor_config=ExecutorConfig(executor_type="FuturesExecutor"),
# )


# # # coffea-casa, FuturesExecutor
# config = RunConfig(
#     strategy="by_dataset",
#     percentage=10,
#     chunk_fraction=0.4,
#     facility=facilities.coffea_casa,
#     executor_config=ExecutorConfig(executor_type="FuturesExecutor"),
# )

# coffea-casa, DaskExecutor (connects to tls://localhost:8786)
config = RunConfig(
    strategy="by_dataset",
    percentage=10,
    chunk_fraction=0.7,
    facility=CoffeaCasaFactory(),
    executor_config=ExecutorConfig(
        executor_type="DaskExecutor",
        worker_packages=("git+https://github.com/hooloobooroodkoo/coffea.git@master",),
        worker_files=("analysis.py",),
    ),
)

result = render(workflow, config)


Workflow DAG:
  [0] Fileset -> Fileset builder=<function get_fileset at 0x7f87ce6b36a0>
  [1] SingleMuonAnalysis -> Analysis builder=<function run_analysis at 0x7f87ce6b37e0>
  [2] PlottingMuonAnalysis -> Plotting builder=analysis:plot_results
Edges:
  Fileset -> SingleMuonAnalysis
  SingleMuonAnalysis -> PlottingMuonAnalysis
Executing step 'Fileset' of type 'Fileset' with the user code <function get_fileset at 0x7f87ce6b36a0> and user parameters {'to_print': '\nTEST:\nparameter testing...\nSUCCESS!\n'}
Extracted from cache: .cache/Fileset/396cfc606caca284dbc25984b2b9ec38f90f85e98b42c872d327b5722e43a234
  -> materialized at .cache/Fileset/396cfc606caca284dbc25984b2b9ec38f90f85e98b42c872d327b5722e43a234
Executing step 'SingleMuonAnalysis' of type 'Analysis' with the user code <function run_analysis at 0x7f87ce6b37e0> and user parameters None
Extracted from cache: .cache/Chunking/c7d4db80e19846c781ca74653656a2abf8234cfd85fb15afa17dac0a8afa0864
Connecting to Dask scheduler...


Output()

Installing on workers: ['git+https://github.com/hooloobooroodkoo/coffea.git@master']
Uploaded analysis.py to workers

Split strategy applied, starting independent processing of 10 fileset subsets...

chunk_fraction=0.7: processing 7 of 10 chunks
------------------------------------
Processing fileset_chunk_0.json
Extracted from cache: .cache/ChunkAnalysis/c9d40f52378c16413e5cc38db2ccada46124698b2fb8e5e31d428de29c54adb1
Successfully processed!
------------------------------------
Processing fileset_chunk_1.json
Extracted from cache: .cache/ChunkAnalysis/828b42b4a82b6117f8ef44a4fc1095a8da17eb420841eb85c5550765acd2b9d6
Successfully processed!
------------------------------------
Processing fileset_chunk_2.json


Output()

╭─────────────────────────────── Traceback (most recent call last) ────────────────────────────────╮
│ /usr/local/lib/python3.12/site-packages/distributed/worker.py:2982 in _run_task_simple           │
│                                                                                                  │
│   2979 │   │   context_meter.meter("thread-cpu", func=thread_time),                              │
│   2980 │   ):                                                                                    │
│   2981 │   │   try:                                                                              │
│ ❱ 2982 │   │   │   result = task(data)                                                           │
│   2983 │   │   except (SystemExit, KeyboardInterrupt):                                           │
│   2984 │   │   │   # Special-case these, just like asyncio does all over the place. They will    │
│   2985 │   │   │   # pass through `fail_hard` and `_handle_stimulus_from_task`, and eventually   │
│                                                                                                  │
│ /usr/local/lib/python3.12/site-packages/dask/_task_spec.py:755 in __call__                       │
│                                                                                                  │
│    752 │   │   │   │   for k, kw in self.kwargs.items()                                          │
│    753 │   │   │   }                                                                             │
│    754 │   │   │   return self.func(*new_argspec, **kwargs)                                      │
│ ❱  755 │   │   return self.func(*new_argspec)                                                    │
│    756 │                                                                                         │
│    757 │   def __setstate__(self, state):                                                        │
│    758 │   │   slots = self.__class__.get_all_slots()                                            │
│                                                                                                  │
│ /usr/local/lib/python3.12/site-packages/coffea/processor/executor.py:221 in __call__             │
│                                                                                                  │
│    218 │   │   return not self.is_ok()                                                           │
│    219 │                                                                                         │
│    220 │   def unwrap(self) -> T:                                                                │
│ ❱  221 │   │   """Return the contained value on ``Ok``; on ``Err`` re-raise the captured except  │
│    222 │   │   raise NotImplementedError                                                         │
│    223                                                                                           │
│    224                                                                                           │
│                                                                                                  │
│ /usr/local/lib/python3.12/site-packages/coffea/processor/executor.py:1173 in automatic_retries   │
│                                                                                                  │
│   1170 │   │   │   determine chunking.  Defaults to a in-memory LRU cache that holds 100k entri  │
│   1171 │   │   │   (about 1MB depending on the length of filenames, etc.)  If you edit an input  │
│   1172 │   │   │   (please don't) during a session, the session can be restarted to clear the c  │
│ ❱ 1173 │   │   checkpointer : CheckpointerABC, optional                                          │
│   1174 │   │   │   A CheckpointerABC instance to manage checkpointing of each chunk output       │
│   1175 │   │   use_result_type : bool, optional                                                  │
│   1176 │   │   │   If True, ``__call__`` returns ``Ok(outpu

TypeError: Runner._work_function() got an unexpected keyword argument 'use_result_type'